In [3]:
"""
PROJECT 1: Smart Sensor Monitor
Simulates temperature and voltage readings, detects anomalies,
and logs data to CSV with colour-coded terminal output.
"""

import random
import csv
from datetime import datetime

# ============================================
# CONFIGURATION
# ============================================

NORMAL_TEMP_MIN = 20.0    # Celsius
NORMAL_TEMP_MAX = 80.0    # Celsius
NORMAL_VOLT_MIN = 3.0     # Volts
NORMAL_VOLT_MAX = 5.5     # Volts

NUM_READINGS = 100         # Number of sensor readings to simulate

# ANSI colour codes for terminal output
COLOR_GREEN = '\033[92m'   # Normal values
COLOR_RED = '\033[91m'     # Anomaly values
COLOR_YELLOW = '\033[93m'  # Headers/warnings
COLOR_RESET = '\033[0m'    # Reset to default

# ============================================
# FUNCTIONS
# ============================================

def generate_sensor_reading():
    """
    Generate a single random temperature and voltage reading.
    Returns: (temperature, voltage) tuple
    """
    temp = random.uniform(15, 90)     # Range 15-90°C (wider than normal)
    voltage = random.uniform(2.5, 6.5) # Range 2.5-6.5V (wider than normal)
    return temp, voltage

def check_anomaly(temp, voltage):
    """
    Check if a reading is anomalous.
    Returns: (is_anomaly, reasons_list)
    """
    is_anomaly = False
    reasons = []
    
    if temp < NORMAL_TEMP_MIN:
        is_anomaly = True
        reasons.append(f"Temp too LOW ({temp:.1f}°C < {NORMAL_TEMP_MIN}°C)")
    elif temp > NORMAL_TEMP_MAX:
        is_anomaly = True
        reasons.append(f"Temp too HIGH ({temp:.1f}°C > {NORMAL_TEMP_MAX}°C)")
    
    if voltage < NORMAL_VOLT_MIN:
        is_anomaly = True
        reasons.append(f"Voltage too LOW ({voltage:.2f}V < {NORMAL_VOLT_MIN}V)")
    elif voltage > NORMAL_VOLT_MAX:
        is_anomaly = True
        reasons.append(f"Voltage too HIGH ({voltage:.2f}V > {NORMAL_VOLT_MAX}V)")
    
    return is_anomaly, reasons

def print_colour_coded(temp, voltage, is_anomaly):
    """
    Print a single reading with colour coding.
    Green = normal, Red = anomaly
    """
    if is_anomaly:
        colour = COLOR_RED
        status = "⚠️ ANOMALY"
    else:
        colour = COLOR_GREEN
        status = "✓ NORMAL"
    
    print(f"{colour}Reading: Temp={temp:6.2f}°C | Volt={voltage:5.2f}V | {status}{COLOR_RESET}")

def save_to_csv(readings, anomaly_count):
    """
    Save all readings to CSV file with timestamp.
    """
    filename = f"sensor_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Write header
        writer.writerow(['Reading_ID', 'Temperature_C', 'Voltage_V', 'Is_Anomaly', 'Anomaly_Reasons'])
        
        for i, reading in enumerate(readings, 1):
            writer.writerow([
                i,
                f"{reading['temp']:.2f}",
                f"{reading['voltage']:.2f}",
                reading['is_anomaly'],
                reading['reasons']
            ])
    
    return filename

def generate_summary_report(readings, anomaly_count):
    """
    Print a summary report of all readings.
    """
    print("\n" + "="*60)
    print(f"{COLOR_YELLOW}📊 SENSOR MONITOR SUMMARY REPORT{COLOR_RESET}")
    print("="*60)
    print(f"Total readings collected: {len(readings)}")
    print(f"Normal readings: {len(readings) - anomaly_count}")
    print(f"{COLOR_RED if anomaly_count > 0 else COLOR_GREEN}Anomalies detected: {anomaly_count}{COLOR_RESET}")
    
    # Anomaly breakdown
    if anomaly_count > 0:
        print("\n--- Anomaly Breakdown ---")
        temp_low = sum(1 for r in readings if r['is_anomaly'] and 'LOW' in r['reasons'] and 'Temp' in r['reasons'])
        temp_high = sum(1 for r in readings if r['is_anomaly'] and 'HIGH' in r['reasons'] and 'Temp' in r['reasons'])
        volt_low = sum(1 for r in readings if r['is_anomaly'] and 'LOW' in r['reasons'] and 'Voltage' in r['reasons'])
        volt_high = sum(1 for r in readings if r['is_anomaly'] and 'HIGH' in r['reasons'] and 'Voltage' in r['reasons'])
        
        print(f"  Temperature too LOW:  {temp_low}")
        print(f"  Temperature too HIGH: {temp_high}")
        print(f"  Voltage too LOW:      {volt_low}")
        print(f"  Voltage too HIGH:     {volt_high}")
    
    # Statistics
    all_temps = [r['temp'] for r in readings]
    all_volts = [r['voltage'] for r in readings]
    
    print("\n--- Statistics ---")
    print(f"Temperature - Min: {min(all_temps):.2f}°C, Max: {max(all_temps):.2f}°C, Avg: {sum(all_temps)/len(all_temps):.2f}°C")
    print(f"Voltage - Min: {min(all_volts):.2f}V, Max: {max(all_volts):.2f}V, Avg: {sum(all_volts)/len(all_volts):.2f}V")
    
    print("="*60)

# ============================================
# MAIN PROGRAM
# ============================================

def main():
    print(f"{COLOR_YELLOW}🔍 SMART SENSOR MONITOR{COLOR_RESET}")
    print(f"Simulating {NUM_READINGS} sensor readings...\n")
    print(f"Normal ranges: Temp {NORMAL_TEMP_MIN}-{NORMAL_TEMP_MAX}°C, Volt {NORMAL_VOLT_MIN}-{NORMAL_VOLT_MAX}V\n")
    
    readings = []
    anomaly_count = 0
    
    for i in range(NUM_READINGS):
        temp, voltage = generate_sensor_reading()
        is_anomaly, reasons = check_anomaly(temp, voltage)
        
        if is_anomaly:
            anomaly_count += 1
            reasons_str = " | ".join(reasons)
        else:
            reasons_str = ""
        
        readings.append({
            'temp': temp,
            'voltage': voltage,
            'is_anomaly': is_anomaly,
            'reasons': reasons_str
        })
        
        # Print first 20 readings (to avoid clutter)
        if i < 20:
            print_colour_coded(temp, voltage, is_anomaly)
    
    if NUM_READINGS > 20:
        print(f"... and {NUM_READINGS - 20} more readings (see CSV for details)")
    
    # Save to CSV
    filename = save_to_csv(readings, anomaly_count)
    print(f"\n💾 Data saved to: {filename}")
    
    # Generate summary report
    generate_summary_report(readings, anomaly_count)
    
    # Print anomaly count at end
    if anomaly_count > 0:
        print(f"\n{COLOR_RED}⚠️ Total anomalies detected: {anomaly_count}{COLOR_RESET}")
    else:
        print(f"\n{COLOR_GREEN}✅ All systems normal! No anomalies detected.{COLOR_RESET}")

if __name__ == "__main__":
    main()

🔍 SMART SENSOR MONITOR
Simulating 100 sensor readings...

Normal ranges: Temp 20.0-80.0°C, Volt 3.0-5.5V

Reading: Temp= 15.56°C | Volt= 4.29V | ⚠️ ANOMALY
Reading: Temp= 27.20°C | Volt= 5.72V | ⚠️ ANOMALY
Reading: Temp= 60.91°C | Volt= 2.70V | ⚠️ ANOMALY
Reading: Temp= 57.41°C | Volt= 5.94V | ⚠️ ANOMALY
Reading: Temp= 53.50°C | Volt= 4.90V | ✓ NORMAL
Reading: Temp= 54.47°C | Volt= 5.11V | ✓ NORMAL
Reading: Temp= 57.21°C | Volt= 5.29V | ✓ NORMAL
Reading: Temp= 59.72°C | Volt= 6.27V | ⚠️ ANOMALY
Reading: Temp= 47.15°C | Volt= 5.93V | ⚠️ ANOMALY
Reading: Temp= 21.69°C | Volt= 4.27V | ✓ NORMAL
Reading: Temp= 24.53°C | Volt= 3.82V | ✓ NORMAL
Reading: Temp= 34.90°C | Volt= 4.18V | ✓ NORMAL
Reading: Temp= 83.53°C | Volt= 2.64V | ⚠️ ANOMALY
Reading: Temp= 82.55°C | Volt= 2.81V | ⚠️ ANOMALY
Reading: Temp= 54.13°C | Volt= 5.89V | ⚠️ ANOMALY
Reading: Temp= 32.14°C | Volt= 3.70V | ✓ NORMAL
Reading: Temp= 39.16°C | Volt= 4.71V | ✓ NORMAL
Reading: Temp= 78.44°C | Volt= 3.90V | ✓ NORMAL
Reading: Tem

In [4]:
"""
PROJECT 2: Electrical Calculations Toolkit
Interactive command-line calculator for electrical engineering formulas.
"""

import math

# ============================================
# FORMULA FUNCTIONS
# ============================================

def ohms_law():
    """Calculate Voltage, Current, or Resistance using V = I × R"""
    print("\n--- Ohm's Law Calculator ---")
    print("What do you want to calculate?")
    print("1. Voltage (V = I × R)")
    print("2. Current (I = V / R)")
    print("3. Resistance (R = V / I)")
    
    choice = input("Enter choice (1-3): ").strip()
    
    if choice == '1':
        current = float(input("Enter Current (I) in Amperes: "))
        resistance = float(input("Enter Resistance (R) in Ohms: "))
        voltage = current * resistance
        print(f"\n✅ Voltage (V) = {current} A × {resistance} Ω = {voltage:.3f} V")
        return voltage
    
    elif choice == '2':
        voltage = float(input("Enter Voltage (V) in Volts: "))
        resistance = float(input("Enter Resistance (R) in Ohms: "))
        if resistance == 0:
            print("❌ Error: Resistance cannot be zero!")
            return None
        current = voltage / resistance
        print(f"\n✅ Current (I) = {voltage} V / {resistance} Ω = {current:.3f} A")
        return current
    
    elif choice == '3':
        voltage = float(input("Enter Voltage (V) in Volts: "))
        current = float(input("Enter Current (I) in Amperes: "))
        if current == 0:
            print("❌ Error: Current cannot be zero!")
            return None
        resistance = voltage / current
        print(f"\n✅ Resistance (R) = {voltage} V / {current} A = {resistance:.3f} Ω")
        return resistance
    
    else:
        print("❌ Invalid choice!")
        return None

def power_calculator():
    """Calculate Electrical Power using P = V × I or P = I² × R or P = V² / R"""
    print("\n--- Power Calculator ---")
    print("Select formula:")
    print("1. Power from Voltage and Current (P = V × I)")
    print("2. Power from Current and Resistance (P = I² × R)")
    print("3. Power from Voltage and Resistance (P = V² / R)")
    
    choice = input("Enter choice (1-3): ").strip()
    
    if choice == '1':
        voltage = float(input("Enter Voltage (V) in Volts: "))
        current = float(input("Enter Current (I) in Amperes: "))
        power = voltage * current
        print(f"\n✅ Power (P) = {voltage} V × {current} A = {power:.3f} W")
        return power
    
    elif choice == '2':
        current = float(input("Enter Current (I) in Amperes: "))
        resistance = float(input("Enter Resistance (R) in Ohms: "))
        power = current**2 * resistance
        print(f"\n✅ Power (P) = {current}² A × {resistance} Ω = {power:.3f} W")
        return power
    
    elif choice == '3':
        voltage = float(input("Enter Voltage (V) in Volts: "))
        resistance = float(input("Enter Resistance (R) in Ohms: "))
        if resistance == 0:
            print("❌ Error: Resistance cannot be zero!")
            return None
        power = voltage**2 / resistance
        print(f"\n✅ Power (P) = {voltage}² V / {resistance} Ω = {power:.3f} W")
        return power
    
    else:
        print("❌ Invalid choice!")
        return None

def rc_time_constant():
    """Calculate RC Time Constant: τ = R × C"""
    print("\n--- RC Time Constant Calculator ---")
    print("Formula: τ = R × C (time to charge/discharge to 63.2%)")
    
    resistance = float(input("Enter Resistance (R) in Ohms: "))
    capacitance = float(input("Enter Capacitance (C) in Farads: "))
    
    tau = resistance * capacitance
    print(f"\n✅ Time Constant (τ) = {resistance} Ω × {capacitance} F = {tau:.6f} s")
    print(f"   → 1τ = {tau:.6f} seconds")
    print(f"   → 5τ (fully charged) = {5*tau:.6f} seconds")
    return tau

def series_resistance():
    """Calculate total resistance for resistors in series: R_total = R1 + R2 + ..."""
    print("\n--- Series Resistance Calculator ---")
    print("Formula: R_total = R1 + R2 + R3 + ...")
    
    resistors = []
    while True:
        try:
            r = input(f"Enter resistance value (Ω) for R{len(resistors)+1} (or 'done' to finish): ")
            if r.lower() == 'done':
                break
            resistors.append(float(r))
        except ValueError:
            print("Please enter a valid number!")
    
    if not resistors:
        print("No resistors entered!")
        return None
    
    total = sum(resistors)
    print(f"\n✅ Total Series Resistance:")
    for i, r in enumerate(resistors, 1):
        print(f"   R{i} = {r} Ω")
    print(f"   R_total = {total} Ω")
    return total

def parallel_resistance():
    """Calculate total resistance for resistors in parallel: 1/R_total = 1/R1 + 1/R2 + ..."""
    print("\n--- Parallel Resistance Calculator ---")
    print("Formula: 1/R_total = 1/R1 + 1/R2 + 1/R3 + ...")
    
    resistors = []
    while True:
        try:
            r = input(f"Enter resistance value (Ω) for R{len(resistors)+1} (or 'done' to finish): ")
            if r.lower() == 'done':
                break
            resistors.append(float(r))
        except ValueError:
            print("Please enter a valid number!")
    
    if not resistors:
        print("No resistors entered!")
        return None
    
    # Calculate parallel resistance
    reciprocal_sum = sum(1/r for r in resistors)
    total = 1 / reciprocal_sum if reciprocal_sum != 0 else 0
    
    print(f"\n✅ Total Parallel Resistance:")
    for i, r in enumerate(resistors, 1):
        print(f"   R{i} = {r} Ω")
    print(f"   R_total = {total:.6f} Ω")
    return total

# ============================================
# MAIN MENU
# ============================================

def display_menu():
    """Display the main menu"""
    print("\n" + "="*50)
    print("⚡ ELECTRICAL CALCULATIONS TOOLKIT")
    print("="*50)
    print("1. Ohm's Law (V = I × R)")
    print("2. Power Calculator (P = V × I)")
    print("3. RC Time Constant (τ = R × C)")
    print("4. Series Resistance (R_total = R1 + R2 + ...)")
    print("5. Parallel Resistance (1/R_total = 1/R1 + 1/R2 + ...)")
    print("6. Help / Formulas")
    print("7. Exit")
    print("="*50)

def show_help():
    """Display formula reference"""
    print("\n" + "="*50)
    print("📖 ELECTRICAL FORMULAS REFERENCE")
    print("="*50)
    print("\nOhm's Law:")
    print("  V = I × R    (Voltage = Current × Resistance)")
    print("  I = V / R    (Current = Voltage / Resistance)")
    print("  R = V / I    (Resistance = Voltage / Current)")
    
    print("\nPower Formulas:")
    print("  P = V × I    (Power = Voltage × Current)")
    print("  P = I² × R   (Power = Current² × Resistance)")
    print("  P = V² / R   (Power = Voltage² / Resistance)")
    
    print("\nRC Circuit:")
    print("  τ = R × C    (Time constant in seconds)")
    
    print("\nResistors in Series:")
    print("  R_total = R1 + R2 + R3 + ...")
    
    print("\nResistors in Parallel:")
    print("  1/R_total = 1/R1 + 1/R2 + 1/R3 + ...")
    print("="*50)

def main():
    print("⚡ Welcome to the Electrical Calculations Toolkit!")
    
    while True:
        display_menu()
        choice = input("\nEnter your choice (1-7): ").strip()
        
        if choice == '1':
            ohms_law()
        elif choice == '2':
            power_calculator()
        elif choice == '3':
            rc_time_constant()
        elif choice == '4':
            series_resistance()
        elif choice == '5':
            parallel_resistance()
        elif choice == '6':
            show_help()
        elif choice == '7':
            print("\n✅ Thank you for using the Electrical Calculations Toolkit! Goodbye!")
            break
        else:
            print("❌ Invalid choice. Please enter 1-7.")
        
        input("\nPress Enter to continue...")

if __name__ == "__main__":
    main()

⚡ Welcome to the Electrical Calculations Toolkit!

⚡ ELECTRICAL CALCULATIONS TOOLKIT
1. Ohm's Law (V = I × R)
2. Power Calculator (P = V × I)
3. RC Time Constant (τ = R × C)
4. Series Resistance (R_total = R1 + R2 + ...)
5. Parallel Resistance (1/R_total = 1/R1 + 1/R2 + ...)
6. Help / Formulas
7. Exit



Enter your choice (1-7):  1



--- Ohm's Law Calculator ---
What do you want to calculate?
1. Voltage (V = I × R)
2. Current (I = V / R)
3. Resistance (R = V / I)


Enter choice (1-3):  1
Enter Current (I) in Amperes:  12
Enter Resistance (R) in Ohms:  2



✅ Voltage (V) = 12.0 A × 2.0 Ω = 24.000 V


...

❌ Invalid choice. Please enter 1-7.



Press Enter to continue... 



⚡ ELECTRICAL CALCULATIONS TOOLKIT
1. Ohm's Law (V = I × R)
2. Power Calculator (P = V × I)
3. RC Time Constant (τ = R × C)
4. Series Resistance (R_total = R1 + R2 + ...)
5. Parallel Resistance (1/R_total = 1/R1 + 1/R2 + ...)
6. Help / Formulas
7. Exit



Enter your choice (1-7):  7



✅ Thank you for using the Electrical Calculations Toolkit! Goodbye!


In [11]:


import csv
import random
from datetime import datetime, timedelta
from collections import defaultdict

# ============================================
# CONFIGURATION
# ============================================

INTERVAL_MINUTES = 5      # Read every 5 minutes
HOURS_IN_DAY = 24
READINGS_PER_HOUR = 60 // INTERVAL_MINUTES  # 12 readings per hour
TOTAL_READINGS = HOURS_IN_DAY * READINGS_PER_HOUR  # 288 readings

# Sensor normal ranges
TEMP_MIN = 18.0
TEMP_MAX = 82.0
HUMIDITY_MIN = 30.0
HUMIDITY_MAX = 90.0

# ============================================
# SIMULATION FUNCTIONS
# ============================================

def generate_sensor_value(min_val, max_val, time_of_day):
    """
    Generate realistic sensor values with daily pattern.
    time_of_day: hour (0-23) to add temperature cycle
    """
    # Add diurnal cycle for temperature (hotter at noon, cooler at night)
    if 'temp' in str(min_val).lower():
        diurnal_shift = 5 * abs(time_of_day - 12) / 12
        base = random.uniform(min_val, max_val)
        return base - diurnal_shift
    else:
        return random.uniform(min_val, max_val)

def simulate_24h_sensor_data():
    """
    Generate 24 hours of sensor readings at 5-minute intervals.
    Returns: list of dictionaries with timestamp, temperature, humidity
    """
    readings = []
    start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
    
    for i in range(TOTAL_READINGS):
        # Calculate current time
        current_time = start_time + timedelta(minutes=i * INTERVAL_MINUTES)
        hour = current_time.hour
        
        # Generate realistic values
        # Temperature: lower at night, higher during day
        if hour < 6 or hour > 20:  # Night
            temp = random.uniform(15, 25)
        elif 6 <= hour < 12:  # Morning
            temp = random.uniform(20, 30)
        elif 12 <= hour < 18:  # Afternoon
            temp = random.uniform(25, 35)
        else:  # Evening
            temp = random.uniform(20, 28)
        
        # Add some random noise
        temp += random.uniform(-2, 2)
        
        # Humidity: often inverse of temperature
        humidity = 80 - (temp - 20) * 1.5 + random.uniform(-5, 5)
        humidity = max(20, min(95, humidity))
        
        readings.append({
            'timestamp': current_time,
            'temperature': round(temp, 2),
            'humidity': round(humidity, 2)
        })
    
    return readings

def save_to_csv(readings, filename="sensor_log.csv"):
    """Save readings to CSV file"""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Timestamp', 'Temperature_C', 'Humidity_%'])
        
        for reading in readings:
            writer.writerow([
                reading['timestamp'].strftime('%Y-%m-%d %H:%M:%S'),
                reading['temperature'],
                reading['humidity']
            ])
    
    print(f"💾 Data saved to {filename}")
    return filename

def read_csv_and_compute_hourly_averages(filename):
    """
    Read CSV file and compute hourly averages.
    Returns: dictionary with hour as key and (avg_temp, avg_humidity) as value
    """
    hourly_temps = defaultdict(list)
    hourly_humidity = defaultdict(list)
    
    with open(filename, 'r') as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        
        for row in reader:
            timestamp_str = row[0]
            temp = float(row[1])
            humidity = float(row[2])
            
            # Extract hour from timestamp
            dt = datetime.strptime(timestamp_str, '%Y-%m-%d %H:%M:%S')
            hour = dt.hour
            
            hourly_temps[hour].append(temp)
            hourly_humidity[hour].append(humidity)
    
    # Compute averages
    averages = {}
    for hour in range(24):
        avg_temp = sum(hourly_temps[hour]) / len(hourly_temps[hour]) if hourly_temps[hour] else 0
        avg_humidity = sum(hourly_humidity[hour]) / len(hourly_humidity[hour]) if hourly_humidity[hour] else 0
        averages[hour] = {
            'temp': round(avg_temp, 2),
            'humidity': round(avg_humidity, 2),
            'readings_count': len(hourly_temps[hour])
        }
    
    return averages

def print_hourly_table(averages):
    """Print formatted table of hourly statistics"""
    print("\n" + "="*70)
    print("📊 HOURLY AVERAGE REPORT")
    print("="*70)
    print(f"{'Hour':<8} {'Temperature (C)':<18} {'Humidity (%)':<15} {'Readings':<10}")
    print("-"*70)
    
    for hour in range(24):
        data = averages[hour]
        # Highlight peak and lowest
        temp_str = f"{data['temp']:>6.1f}°C"
        humidity_str = f"{data['humidity']:>6.1f}%"
        
        if hour == 12 or hour == 13:  # Typical hottest hours
            temp_str = f"🔥{temp_str}🔥"
        
        print(f"{hour:02d}:00-{hour:02d}:59  {temp_str:<18}  {humidity_str:<15}  {data['readings_count']:<10}")
    
    print("="*70)
    
    # Find hottest and coldest hours
    temps = [(hour, data['temp']) for hour, data in averages.items()]
    hottest = max(temps, key=lambda x: x[1])
    coldest = min(temps, key=lambda x: x[1])
    
    print(f"\n🌡️  Hottest hour: {hottest[0]:02d}:00 ({hottest[1]:.1f}°C)")
    print(f"❄️  Coldest hour: {coldest[0]:02d}:00 ({coldest[1]:.1f}°C)")

def plot_ascii_trend(averages):
    """Simple ASCII bar chart of temperature trend"""
    print("\n" + "="*70)
    print("📈 TEMPERATURE TREND (ASCII Chart)")
    print("="*70)
    
    temps = [averages[h]['temp'] for h in range(24)]
    min_temp = min(temps)
    max_temp = max(temps)
    
    # Scale to 20-30 characters for display
    scale = 25 / (max_temp - min_temp) if max_temp != min_temp else 1
    
    for hour in range(24):
        temp = temps[hour]
        bar_length = int((temp - min_temp) * scale)
        bar = '█' * bar_length if bar_length > 0 else '·'
        
        # Mark peak
        marker = " 🔥" if temp == max_temp else " ❄️" if temp == min_temp else ""
        print(f"{hour:02d}:00 | {bar:<30} {temp:5.1f}°C{marker}")
    
    print("="*70)

# ============================================
# MAIN PROGRAM
# ============================================

def main():
    print("="*60)
    print("📟 DATA LOGGER SIMULATOR")
    print("="*60)
    print(f"Simulating 24 hours of sensor data")
    print(f"Reading interval: {INTERVAL_MINUTES} minutes")
    print(f"Total readings: {TOTAL_READINGS}")
    print("="*60)
    
    # Step 1: Simulate data
    print("\n🔄 Simulating sensor readings...")
    readings = simulate_24h_sensor_data()
    print(f"✅ Generated {len(readings)} readings")
    
    # Step 2: Save to CSV
    filename = save_to_csv(readings)
    
    # Step 3: Read back and compute averages
    print("\n📖 Reading CSV and computing hourly averages...")
    averages = read_csv_and_compute_hourly_averages(filename)
    
    # Step 4: Display results
    print_hourly_table(averages)
    plot_ascii_trend(averages)
    
    # Additional statistics
    all_temps = [r['temperature'] for r in readings]
    all_humidity = [r['humidity'] for r in readings]
    
    print("\n" + "="*60)
    print("📈 24-HOUR SUMMARY STATISTICS")
    print("="*60)
    print(f"Temperature - Min: {min(all_temps):.1f}°C | Max: {max(all_temps):.1f}°C | Avg: {sum(all_temps)/len(all_temps):.1f}°C")
    print(f"Humidity    - Min: {min(all_humidity):.1f}% | Max: {max(all_humidity):.1f}% | Avg: {sum(all_humidity)/len(all_humidity):.1f}%")
    print("="*60)

if __name__ == "__main__":
import csv
import random
from datetime import datetime, timedelta

# Create sample sensor data
def create_sample_csv(filename="sensor_data.csv"):
    """Generate sample sensor readings and save to CSV"""
    
    print(f"🔄 Creating sample data file: {filename}")
    
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # Write header
        writer.writerow(['Timestamp', 'Temperature_C', 'Voltage_V', 'Humidity_%'])
        
        # Generate 100 readings over 24 hours
        start_time = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        
        for i in range(100):
            current_time = start_time + timedelta(minutes=i*15)  # Every 15 minutes
            hour = current_time.hour
            
            # Temperature: hotter at noon (hour 12), cooler at night
            temp = 25 + 10 * (1 - abs(hour - 12) / 12) + random.uniform(-3, 3)
            temp = round(max(15, min(45, temp)), 2)  # Keep between 15-45°C
            
            # Voltage: normal 3.0-5.5V, with occasional anomalies
            if random.random() < 0.08:  # 8% chance of anomaly
                voltage = round(random.uniform(2.0, 7.0), 2)  # Outside normal range
            else:
                voltage = round(random.uniform(3.2, 5.2), 2)  # Normal range
            
            # Humidity: inversely related to temperature
            humidity = 80 - (temp - 20) * 1.2 + random.uniform(-8, 8)
            humidity = round(max(20, min(95, humidity)), 2)  # Keep between 20-95%
            
            writer.writerow([
                current_time.strftime('%Y-%m-%d %H:%M:%S'),
                temp,
                voltage,
                humidity
            ])
    
    print(f"✅ Sample data saved to {filename}")
    print(f"   Total records: 100")
    print(f"   Columns: Timestamp, Temperature_C, Voltage_V, Humidity_%")
    return filename

# Create the CSV file
csv_filename = create_sample_csv("sensor_data.csv")    main()

IndentationError: expected an indented block after 'if' statement on line 223 (1554317125.py, line 224)

In [10]:
📁 File 'sensor_data.csv' not found. Generating sample data...

🔄 Generating sample data to sensor_data.csv...
✅ Sample data generated with 100 readings
   Includes some anomalies for testing

🔄 Loading data from sensor_data.csv...
✅ Loaded 100 records with 4 columns

================================================================================
📊 SYSTEM STATUS DASHBOARD
   File: sensor_data.csv
================================================================================

📁 FILE INFORMATION
   Total rows: 100
   Total columns: 4
   Columns: timestamp, temperature_c, voltage_v, humidity_%

📈 COLUMN STATISTICS
--------------------------------------------------------------------------------

📌 TIMESTAMP
   ──────────────────────────────────────────────────
   ⚠️  No numeric data found in this column

📌 TEMPERATURE_C
   ──────────────────────────────────────────────────
   ✓ Valid readings: 100/100
   ✓ Min: 16.2345
   ✓ Max: 38.9123
   ✓ Mean: 25.6789
   ✅ No anomalies detected

  Distribution of temperature_c:
  Range                | Frequency  | Chart
  --------------------+------------+-------------------------
  16.23 - 18.56       | 5          | ████
  18.56 - 20.89       | 8          | ██████
  20.89 - 23.22       | 15         | ███████████
  ...

================================================================================
📊 SUMMARY DASHBOARD
================================================================================

📋 OVERALL STATISTICS:
   • Total columns analyzed: 3
   • Total anomalies found: 8
   • Data quality: 97.3% valid

⚠️  WARNING: Anomalies detected! Review the data above.
   Suggested actions:
   1. Check sensor calibration
   2. Verify normal range thresholds
   3. Investigate timestamp of anomalies

================================================================================

IndentationError: unindent does not match any outer indentation level (<string>, line 35)